# Lung Dataset Structure Analysis

Explores the lung atlas h5ad without loading the full X matrix (~32 GB sparse).

In [ ]:
import h5py
import numpy as np
import pandas as pd
import scanpy as sc

LUNG_PATH = '/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung.h5ad'
print("Path:", LUNG_PATH)


## 1. Dataset Dimensions

In [ ]:
with h5py.File(LUNG_PATH, 'r') as f:
    X = f['X']
    indptr = X['indptr'][:]
    n_cells = len(indptr) - 1
    n_nnz   = len(X['data'])
    n_genes = len(f['var']['_index'])
    data_dtype = X['data'].dtype

print(f"Cells   : {n_cells:,}")
print(f"Genes   : {n_genes:,}")
print(f"Non-zeros: {n_nnz:,}")
print(f"Density  : {n_nnz / (n_cells * n_genes) * 100:.4f}%")
print(f"X dtype  : {data_dtype}")
print()

# Memory estimates
nnz_float32 = n_nnz * 4          # data
nnz_int32   = n_nnz * 4          # indices
indptr_int64 = (n_cells + 1) * 8  # indptr
total_bytes = nnz_float32 + nnz_int32 + indptr_int64
print(f"Estimated in-memory size (CSR float32): {total_bytes / 1e9:.1f} GB")

# Per-cell sparsity
nnz_per_cell = np.diff(indptr)
print(f"Non-zeros per cell: mean={nnz_per_cell.mean():.0f}, median={np.median(nnz_per_cell):.0f}, max={nnz_per_cell.max()}")


## 2. Obs Columns

In [ ]:
with h5py.File(LUNG_PATH, 'r') as f:
    obs_keys = list(f['obs'].keys())

print(f"Total obs columns: {len(obs_keys)}")
print("All columns:")
for k in sorted(obs_keys):
    print(f"  {k}")


## 3. Annotation Levels

In [ ]:
# Load obs metadata only (fast — no X loaded)
adata = sc.read_h5ad(LUNG_PATH, backed='r')
obs = adata.obs

ANN_COLS = ['ann_level_1', 'ann_level_2', 'ann_level_3',
            'ann_level_4', 'ann_level_5', 'ann_finest_level']

for col in ANN_COLS:
    if col not in obs.columns:
        print(f"{col}: NOT PRESENT")
        continue
    n_unique  = obs[col].nunique()
    n_missing = obs[col].isna().sum()
    print(f"{col:20s}: {n_unique:4d} unique, {n_missing:7,} missing ({n_missing/len(obs)*100:.1f}%)")


## 4. Cell Type Counts per Level

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, col in zip(axes, ANN_COLS):
    if col not in obs.columns:
        ax.set_visible(False)
        continue
    vc = obs[col].value_counts().head(30)
    vc.plot(kind='barh', ax=ax)
    ax.set_title(f'{col}\n({obs[col].nunique()} unique)')
    ax.set_xlabel('Cell count')
    ax.tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.show()


## 5. Finest-Level Distribution (Top 40)

In [ ]:
vc = obs['ann_finest_level'].value_counts()
print(f"Total finest-level cell types: {len(vc)}")
print()
print("Top 40:")
print(vc.head(40).to_string())
print()
print("Bottom 10 (rarest):")
print(vc.tail(10).to_string())


## 6. Hierarchy Structure Preview

In [ ]:
LEVEL_COLS = ['ann_level_1', 'ann_level_2', 'ann_level_3',
              'ann_level_4', 'ann_level_5']
ANN_COL    = 'ann_finest_level'

def is_valid(v):
    if pd.isna(v): return False
    return str(v).strip().lower() not in ('', 'nan', 'none', 'unknown', 'na', 'n/a')

# Build ontology
ontology_dict = {}
cols_ordered = LEVEL_COLS + [ANN_COL]

for k in range(1, len(cols_ordered)):
    pc, cc = cols_ordered[k-1], cols_ordered[k]
    if pc not in obs.columns or cc not in obs.columns:
        continue
    pairs = obs[[pc, cc]].dropna().drop_duplicates()
    for _, row in pairs.iterrows():
        p, ch = str(row[pc]), str(row[cc])
        if is_valid(p) and is_valid(ch) and ch not in ontology_dict:
            ontology_dict[ch] = p

for val in obs[cols_ordered[0]].dropna().unique():
    if is_valid(val) and str(val) not in ontology_dict:
        ontology_dict[str(val)] = None

print(f"Ontology entries : {len(ontology_dict)}")
interior = {p for p in ontology_dict.values() if p is not None}
leaves   = [n for n in ontology_dict if n not in interior]
print(f"Interior nodes   : {len(interior)}")
print(f"Leaf nodes       : {len(leaves)}")
print()

# Check for collisions (nodes used as both leaf and interior)
finest_vals = set(obs[ANN_COL].dropna().astype(str).unique())
collisions  = sorted(interior & finest_vals)
print(f"Collision labels (interior + finest-level): {len(collisions)}")
if collisions:
    for c in collisions[:10]:
        children = [n for n, p in ontology_dict.items() if p == c]
        print(f"  '{c}' — {len(children)} children")


## 7. Sample Rows from X (Gene Expression Preview)

In [ ]:
# Read a few rows directly from backed X — no full matrix load
# gene_names come from adata.var (already in memory from the backed read above)
# so we avoid slicing any h5py Group objects.

def _read_var_col(f, col):
    """Safely read a var column that may be a plain Dataset or a categorical Group."""
    import h5py
    node = f["var"][col]
    if isinstance(node, h5py.Group):
        # Categorical: stored as {categories, codes}
        cats  = node["categories"][:]
        codes = node["codes"][:]
        vals  = cats[codes]
    else:
        vals = node[:]
    return np.array([v.decode() if isinstance(v, bytes) else str(v) for v in vals])

# Use adata.var_names — guaranteed correct, no h5py Group ambiguity
gene_names = np.array(adata.var_names)

with h5py.File(LUNG_PATH, "r") as f:
    h5_X   = f["X"]
    indptr = h5_X["indptr"][:]

    print("Sample cells (first 5):")
    for row_i in range(5):
        start, end = int(indptr[row_i]), int(indptr[row_i + 1])
        vals = h5_X["data"][start:end]
        cols = h5_X["indices"][start:end]
        order = np.argsort(vals)[-10:][::-1]
        top_genes = [(gene_names[cols[j]], float(vals[j])) for j in order if vals[j] > 0]
        ct = obs["ann_finest_level"].iloc[row_i]
        print(f"  Cell {row_i} ({ct}): {top_genes[:5]}")


## 8. Other Useful Metadata Columns

In [ ]:
useful_cols = ['tissue', 'disease', 'sex', 'study',
               'lung_condition', 'assay', 'donor_id']
for col in useful_cols:
    if col in obs.columns:
        vc = obs[col].value_counts()
        print(f"{col} ({vc.shape[0]} unique): {list(vc.index[:5])} ...")
    else:
        print(f"{col}: NOT PRESENT")
